# Solution D2 · Impurity importances reward noise: comparison with permutation

**Type**: Interpret

## Objective

Add to the Titanic two columns **with no information** (a continuous random number and a random ticket code with 100,000 values) and compare the impurity importances (`feature_importances_`, computed on the training set) with those by **permutation** on the test set: the first gives the two noise columns a large share, the second
brings them to zero.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

df = ds.titanic()
X, y = df.drop(columns="survived").copy(), df["survived"]
r = np.random.default_rng(0)
X["random_number"] = r.uniform(size=len(X))
X["ticket_code"] = r.integers(0, 100000, len(X))
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare", "random_number", "ticket_code"]
CAT = ["sex", "port", "deck", "title"]
names = NUM + CAT
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


def build(num, cat):
    prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), num), ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder()), cat)])
    return make_pipeline(prep, DecisionTreeClassifier(min_samples_leaf=5, random_state=0))


pipe = build(NUM, CAT).fit(X_tr, y_tr)

## Commented solution

In [ ]:
imp = pd.Series(pipe[-1].feature_importances_, index=names)
result = permutation_importance(pipe, X_te, y_te, n_repeats=20, random_state=0, scoring="roc_auc")
perm = pd.Series(result.importances_mean, index=X_te.columns).reindex(names)
pd.DataFrame({"impurity (training)": imp, "permutation (test)": perm}).round(3)

In [ ]:
top5_impurity = list(imp.nlargest(5).index)
top5_permutation = list(perm.nlargest(5).index)
print("top 5 by impurity:", top5_impurity, "\ntop 5 by permutation:", top5_permutation)

In [ ]:
noise = ["random_number", "ticket_code"]
noise_share_impurity = float(imp[noise].sum())
positive = perm.clip(lower=0)
noise_share_permutation = float(positive[noise].sum() / positive.sum())
print(f"share of the noise: impurity {noise_share_impurity:.1%} | permutation {noise_share_permutation:.1%}")

In [ ]:
test_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
without = build([c for c in NUM if c not in noise], CAT).fit(X_tr, y_tr)
auc_without = roc_auc_score(y_te, without.predict_proba(X_te)[:, 1])
print(f"test AUC with the two useless columns: {test_auc:.3f} | without: {auc_without:.3f}")

## Check

In [ ]:
with verify():
    check(list(imp.index) == names and close(imp.sum(), 1.0) and list(perm.index) == names, "the two series have the same names in the same order; the impurity importances sum to 1", "use X_te[names] or reindex(names) for the permutation")
    check("sex" in top5_impurity and "sex" in top5_permutation and top5_impurity[0] == "sex" and top5_permutation[0] == "sex", "`sex` is at the top of both", "the most important feature")
    check("random_number" in top5_impurity and "random_number" not in top5_permutation, "`random_number` is in the top 5 by impurity but not by permutation", "compare the two lists")
    check(noise_share_impurity > 0.15 and noise_share_permutation < 0.06, "the noise is worth over 15% of the impurity importance and less than 6% of the permutation one", "sum of the two useless features")
    check(auc_without is not None and auc_without > test_auc - 0.03, "without the useless columns the AUC does not get worse (and often improves)", "build(NUM without noise, CAT)")
done()

## What you learned

- **The impurity importance gives 18% to the noise:** `random_number` (10.5%) is more important than `siblings_spouses` and on par with `age`, and `ticket_code` (7.7%) is sixth. Neither column contains information: the tree uses them to **memorize** the training cases. It is a known flaw of trees: features with
  many distinct values offer more thresholds to try and more chances of finding a lucky one.
- **The permutation on the test set puts them back where they belong** (contribution 0 or slightly negative: shuffling a useless column does not make anything worse, or improves by chance). It measures how much the AUC **on new data** depends on that column: it is the operational definition of "useful".
- The model with the two useless columns is a little **worse** than the one without (the noise increases the variance): another reason not to trust `feature_importances_` as a feature selection tool.
- Rule: use the impurity importance for a quick first look, the permutation importance (on data not used for training) for the conclusions, and remember that with correlated features even the permutation splits the importance among the copies (it underestimates all of them).

## Common pitfall

Selecting features by keeping those with an impurity importance above a threshold. Random columns with many values exceed the threshold, and informative ones with few distinct values (booleans) can stay below it.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why can the feature_importances_ of a random forest be misleading?</summary>

Because it is computed on the training set and with the impurity: features with many distinct values (continuous, identifiers) have more candidate thresholds and more chances of reducing the impurity by chance, so they receive importance even if they are noise. I use the permutation importance on a validation set, which measures
how much the performance gets worse by shuffling the column, or SHAP; and I keep in mind that correlated features share the importance.

</details>